# Analysis 1 — Treasury Market Conditions and Scenarios

## Setup — FRED observations

In [ ]:
from pathlib import Path
from bondview.data.fred import FredDataManager
from bondview.presentation.widgets import DateRangePicker
from bondview import TreasuryMarketAnalysis

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SNAPSHOT_DIR = PROJECT_ROOT / "data/fred"
fred = FredDataManager(SNAPSHOT_DIR, env_path=PROJECT_ROOT / ".env")

### Donwload

In [ ]:
FRED_SERIES = (
    "DGS2", "DGS10", "DGS30",
    "THREEFY2", "THREEFYTP2", "THREEFY10", "THREEFYTP10",
)
start = None
end = None
downloaded = fred.download(series_ids=FRED_SERIES, start=start, end=end)

In [ ]:
# Optional inspection; downloaded is also a DataFrame.
#fred.report(series_ids='DGS10')  
fred.report()

In [ ]:
fred.save()

### Load Treasury observations

Required data setup for A and C; no KW inputs are needed.

In [ ]:
observations = fred.load(series_ids=["DGS2", "DGS10", "DGS30"])
fred.report()
market = TreasuryMarketAnalysis(observations)

## A. Yield-curve conditions and movements

Examine 2Y, 10Y, and 30Y Treasury yields and compare available observations within the selected date range.

### Treasury yield history

Plot historical 2Y, 10Y, and 30Y Treasury yields over the selected date range.

In [ ]:
date_range = DateRangePicker(observations)

In [ ]:
date_range

In [ ]:
history_figure = market.show_history(*date_range.get_dates())

### Treasury yield and curve movements

Calculate yield and spread changes between selected dates, classify curve movements, and display the results

In [ ]:
curve = market.curve(
    *date_range.get_dates(),
    yield_change_tolerance_bp=5,
    slope_tolerance_bp=5,
)
curve.show()

## B. Expected short rates and term premiums

*Load KW model estimates*

In [ ]:
kw_observations = fred.load(
    series_ids=["THREEFY2", "THREEFYTP2", "THREEFY10", "THREEFYTP10"]
)

### 2Y and 10Y yield decomposition

In [ ]:
kw_date_range = DateRangePicker(kw_observations)

In [ ]:
kw_date_range

In [ ]:
decomposition = market.decomposition(kw_observations, *kw_date_range.get_dates())
decomposition_figure = decomposition.show(width=1300, height=600)

## C. Conditional joint yield movements

Run Setup (load Treasury observations) and this section independently of A/B. Select whole historical intervals matching an inclusive 2Y or 10Y change range.

In [ ]:
condition_tenor = "DGS10"
lower_bp, upper_bp = -25, 25
horizon_months = 6
sampling = "monthly"  # Or "daily"
reference_start = None  # Earliest complete common observation
reference_end = None    # Latest complete common observation

### Complete scenario set

Monthly intended ends move backward by one calendar month from the latest complete observation, always offset from that original anchor. Daily intended ends use every complete date, newest first. Subtract the calendar-month horizon from each intended end and select the first and last complete observations inside the inclusive window. Stop before intended starts precede complete reference history. Skip windows with fewer than two observations or more than seven days of combined endpoint shortfall. No outside-window searching or yield filling is used. Actual/intended dates, elapsed days and shortfall remain available for audit.

In [ ]:
scenarios = market.scenarios(
    condition_tenor=condition_tenor,
    lower_bp=lower_bp, upper_bp=upper_bp,
    horizon_months=horizon_months, sampling=sampling,
    reference_start=reference_start, reference_end=reference_end,
)
joint_scenarios = scenarios.scenarios
scenarios

### Descriptive statistical summary

Marginal quantiles summarize the set; they do not form additional joint scenarios. Overlapping-window counts are descriptive, not independent event frequencies or forecast probabilities.

In [ ]:
scenario_summary = scenarios.show_summary()

### Joint yield changes

Color identifies the conditioning tenor. Hover retains historical interval dates and elapsed days.

In [ ]:
scenario_figure = scenarios.show_plot()